# 01 — Bronze Layer
Ingests 12 monthly NYC Yellow Taxi 2023 Parquet files into Delta Lake.

January files came in as INT64 while Feb-Dec used INT32 — caught this when 92% of rows
showed as nulls. Fixed by reading each file separately and casting to consistent types.

In [0]:
# ADLS auth — credentials pulled from Key Vault at runtime
# Spark configured to use OAuth 2.0 with service principal
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

print("Setup complete")

In [0]:
# Read each monthly file independently to avoid schema conflicts
# January 2023 used INT64 for location IDs, rest used INT32
# Casting to consistent types before union

from pyspark.sql.functions import current_timestamp, lit, col

dfs = []

for month in range(1, 13):
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    path = f"{RAW_PATH}/yellow_taxi/{filename}"
    
    # Read raw parquet — each file read independently (avoids schema conflict)
    df = spark.read.parquet(path)
    
    # Cast all potentially conflicting columns to consistent types
    df_cast = (df
        .withColumn("VendorID",            col("VendorID").cast("long"))
        .withColumn("passenger_count",     col("passenger_count").cast("double"))
        .withColumn("RatecodeID",          col("RatecodeID").cast("double"))
        .withColumn("PULocationID",        col("PULocationID").cast("long"))
        .withColumn("DOLocationID",        col("DOLocationID").cast("long"))
        .withColumn("payment_type",        col("payment_type").cast("long"))
        # Metadata columns
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file",         lit(filename))
        .withColumn("pipeline_name",       lit("nyc_taxi_bronze"))
    )
    
    dfs.append(df_cast)
    print(f"{filename} — schema normalized")

print(f"\nAll 12 months ready")

In [0]:
# Union all 12 months into single DataFrame
from functools import reduce
from pyspark.sql import DataFrame

df_bronze = reduce(DataFrame.union, dfs)
print(f"All months unioned")
print(f"Columns: {len(df_bronze.columns)}")
df_bronze.printSchema()

In [0]:
# Write to Delta Lake — Bronze layer
# partitionBy source_file preserves lineage back to original monthly file
bronze_path = f"{RAW_PATH}/delta/bronze_yellow_taxi"

(df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("source_file")      # partition by file = easy lineage
    .save(bronze_path)
)

print("Bronze Delta table written")

In [0]:
# Quick sanity check — row count and null check on key columns
from pyspark.sql.functions import count, when

df_verify = spark.read.format("delta").load(f"{RAW_PATH}/delta/bronze_yellow_taxi")

print(f"Total rows:    {df_verify.count():,}")
print(f"Total columns: {len(df_verify.columns)}")

# Null check
df_verify.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["VendorID", "passenger_count", "PULocationID", "DOLocationID"]
]).show()

df_verify.show(3)